In [ ]:
! which python

# Homework 6

- All theoretical questions must be answered in your own words, do not copy-paste text from the internet. Points can be deducted for terrible formatting or incomprehensible English.

- Code must be commented. If you use code you found online, you have to add the link to the source you used. There is no penalty for using outside sources as long as you convince us you understand the code.

**To pass the homework you need to complete 100% of the homework**

**Once completed zip the entire directory containing this exercise and upload it to Moodle.**

## Convolutional Neural networks

So far we have worked with deep fully-connected networks, using them to explore different optimization strategies and network architectures. Fully-connected networks are a good approach for experimentation because they are very computationally efficient, but in practice all state-of-the-art results use convolutional networks instead.

First you will implement several layer types that are used in convolutional networks. You will then use these layers to train a convolutional network on the CIFAR-10 dataset. For background reading see http://cs231n.github.io/convolutional-networks/.

**NB! For this task you need to copy `layers.py` from previous homework to current directory.**

In [ ]:
import torch

from data_utils import get_CIFAR10_data

from conv_layers import *

from gradient_check import eval_numerical_gradient_array, eval_numerical_gradient

from cnn import *

from solver import Solver

import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

def rel_error(x, y):

    """ returns relative error """

    x = x.to(torch.float64)
    y = y.to(torch.float64)
    
    return torch.max(torch.abs(x - y) / (torch.max(torch.tensor(1e-8, dtype=torch.float64), torch.abs(x) + torch.abs(y))))

In [ ]:
# set default tensor type
torch.set_default_dtype(torch.float64)

# Load the (preprocessed) CIFAR10 data.
data = get_CIFAR10_data('../cifar-10-batches-py')

for k, v in list(data.items()):
    
    data[k] = torch.tensor(v, dtype=torch.float64)
    print('%s: ' % k, data[k].shape)

# Convolution: Naive forward pass

**Task 6.1** 

The core of a convolutional network is the convolution operation. In the file `conv_layers.py`, implement the forward pass for the convolution layer in the function `conv_forward_naive`. 

You don't have to worry too much about efficiency at this point; just write the code in whatever way you find most clear.

You can test your implementation by running the following:

In [ ]:
x_shape = (2, 3, 4, 4)
w_shape = (3, 3, 4, 4)

x = torch.linspace(-0.1, 0.5, steps=torch.prod(torch.tensor(x_shape)), dtype=torch.float32).reshape(x_shape)
w = torch.linspace(-0.2, 0.3, steps=torch.prod(torch.tensor(w_shape)), dtype=torch.float32).reshape(w_shape)
b = torch.linspace(-0.1, 0.2, steps=3, dtype=torch.float32)

conv_param = {'stride': 2, 'pad': 1}

out, _ = conv_forward_naive(x, w, b, conv_param)

correct_out = torch.tensor([
    [[[-0.08759809, -0.10987781], [-0.18387192, -0.2109216 ]],
     [[ 0.21027089,  0.21661097], [ 0.22847626,  0.23004637]],
     [[ 0.50813986,  0.54309974], [ 0.64082444,  0.67101435]]],
    [[[-0.98053589, -1.03143541], [-1.19128892, -1.24695841]],
     [[ 0.69108355,  0.66880383], [ 0.59480972,  0.56776003]],
     [[ 2.36270298,  2.36904306], [ 2.38090835,  2.38247847]]]
    ], dtype=torch.float64)

diff_ = rel_error(out, correct_out)

print()
assert diff_ < 1e-7, f'Task 6.1 failed: difference too large ({diff_:.6e})'

# Compare your output to ours; difference should be around 10e-8
print('Task 6.1 passed!')
print('conv_forward_naive difference: ', diff_)

# Aside: Image processing via convolutions

As fun way to both check your implementation and gain a better understanding of the type of operation that convolutional layers can perform, we will set up an input containing two images and manually set up filters that perform common image processing operations (grayscale conversion and edge detection). The convolution forward pass will apply these operations to each of the input images. We can then visualize the results as a sanity check.

In [ ]:
import numpy as np
from skimage.transform import resize as imresize
from matplotlib.pyplot import imread

kitten, puppy = imread('kitten.jpg'), imread('puppy.jpg')
# kitten is wide, and puppy is already square
d = kitten.shape[1] - kitten.shape[0]
kitten_cropped = kitten[:, d//2:-d//2, :]

img_size = 200   # Make this smaller if it runs too slow
x = np.zeros((2, 3, img_size, img_size))
x[0, :, :, :] = imresize(puppy, (img_size, img_size)).transpose((2, 0, 1))
x[1, :, :, :] = imresize(kitten_cropped, (img_size, img_size)).transpose((2, 0, 1))

# Set up a convolutional weights holding 2 filters, each 3x3
w = np.zeros((2, 3, 3, 3))

# The first filter converts the image to grayscale.
# Set up the red, green, and blue channels of the filter.
w[0, 0, :, :] = [[0, 0, 0], [0, 0.3, 0], [0, 0, 0]]
w[0, 1, :, :] = [[0, 0, 0], [0, 0.6, 0], [0, 0, 0]]
w[0, 2, :, :] = [[0, 0, 0], [0, 0.1, 0], [0, 0, 0]]

# Second filter detects horizontal edges in the blue channel.
w[1, 2, :, :] = [[1, 2, 1], [0, 0, 0], [-1, -2, -1]]

# Vector of biases. We don't need any bias for the grayscale
# filter, but for the edge detection filter we want to add 128
# to each output so that nothing is negative.
b = np.array([0, 128])

x = torch.from_numpy(x)
w = torch.from_numpy(w)
b = torch.from_numpy(b)

# Compute the result of convolving each input in x with each filter in w,
# offsetting by b, and storing the results in out.
out, _ = conv_forward_naive(x, w, b, {'stride': 1, 'pad': 1})

out = out.cpu().numpy()

def imshow_noax(img, normalize=True):
    """ Tiny helper to show images as uint8 and remove axis labels """
    if normalize:
        img_max, img_min = np.max(img), np.min(img)
        img = 255.0 * (img - img_min) / (img_max - img_min)
    plt.imshow(img.astype('uint8'))
    plt.gca().axis('off')

# Show the original images and the results of the conv operation
plt.subplot(2, 3, 1)
imshow_noax(puppy, normalize=False)
plt.title('Original image')
plt.subplot(2, 3, 2)
imshow_noax(out[0, 0])
plt.title('Grayscale')
plt.subplot(2, 3, 3)
imshow_noax(out[0, 1])
plt.title('Edges')
plt.subplot(2, 3, 4)
imshow_noax(kitten_cropped, normalize=False)
plt.subplot(2, 3, 5)
imshow_noax(out[1, 0])
plt.subplot(2, 3, 6)
imshow_noax(out[1, 1])
plt.show()

**Task 6.2** 

The edge detector filter seems to focus on horizontal lines. What would need to be changed for it to focus on vertical lines? Copy-paste the above code to a new cell and make the change.

**Your Answer:**

In [ ]:
import numpy as np
from skimage.transform import resize as imresize
from matplotlib.pyplot import imread

kitten, puppy = imread('kitten.jpg'), imread('puppy.jpg')
# kitten is wide, and puppy is already square
d = kitten.shape[1] - kitten.shape[0]
kitten_cropped = kitten[:, d//2:-d//2, :]

img_size = 200   # Make this smaller if it runs too slow
x = np.zeros((2, 3, img_size, img_size))
x[0, :, :, :] = imresize(puppy, (img_size, img_size)).transpose((2, 0, 1))
x[1, :, :, :] = imresize(kitten_cropped, (img_size, img_size)).transpose((2, 0, 1))

# Convolution: Naive backward pass

**Task 6.3** 

Implement the backward pass for the convolution operation in the function `conv_backward_naive` in the file `conv_layers.py`. Again, you don't need to worry too much about computational efficiency.

When you are done, run the following to check your backward pass with a numeric gradient check.

In [ ]:
torch.manual_seed(231)

x = torch.randn(4, 3, 5, 5, dtype=torch.float64, requires_grad=False)
w = torch.randn(2, 3, 3, 3, dtype=torch.float64, requires_grad=False)
b = torch.randn(2, dtype=torch.float64, requires_grad=False)

dout = torch.randn(4, 2, 5, 5, dtype=torch.float64, requires_grad=False)
conv_param = {'stride': 1, 'pad': 1}

dx_num = eval_numerical_gradient_array(lambda x_: conv_forward_naive(x_, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w_: conv_forward_naive(x, w_, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b_: conv_forward_naive(x, w, b_, conv_param)[0], b, dout)

out, cache = conv_forward_naive(x, w, b, conv_param)
dx, dw, db = conv_backward_naive(dout, cache)

dx_diff = rel_error(dx, dx_num)
dw_diff = rel_error(dw, dw_num)
db_diff = rel_error(db, db_num)

print()
assert dx_diff < 1e-8, f'Task 6.3 failed: dx error too large ({dx_diff:.6e})'
assert dw_diff < 1e-8, f'Task 6.3 failed: dw error too large ({dw_diff:.6e})'
assert db_diff < 1e-8, f'Task 6.3 failed: db error too large ({db_diff:.6e})'

# Your errors should be around 1e-8'
print('Task 6.3 passed!')
print('dx error: ', rel_error(dx, dx_num).item())
print('dw error: ', rel_error(dw, dw_num).item())
print('db error: ', rel_error(db, db_num).item())

# Max pooling: Naive forward

**Task 6.4** 

Implement the forward pass for the max-pooling operation in the function `max_pool_forward_naive` in the file `conv_layers.py`. Again, don't worry too much about computational efficiency.

Check your implementation by running the following:

In [ ]:
x_shape = (2, 3, 4, 4)

x = torch.linspace(-0.3, 0.4, steps=torch.prod(torch.tensor(x_shape)), dtype=torch.float64).reshape(x_shape)
pool_param = {'pool_width': 2, 'pool_height': 2, 'stride': 2}

out, _ = max_pool_forward_naive(x, pool_param)

correct_out = torch.tensor([
    [[[-0.26315789, -0.24842105], [-0.20421053, -0.18947368]],
     [[-0.14526316, -0.13052632], [-0.08631579, -0.07157895]],
     [[-0.02736842, -0.01263158], [ 0.03157895,  0.04631579]]],
    [[[ 0.09052632,  0.10526316], [ 0.14947368,  0.16421053]],
     [[ 0.20842105,  0.22315789], [ 0.26736842,  0.28210526]],
     [[ 0.32631579,  0.34105263], [ 0.38526316,  0.4       ]]]
     ], dtype=torch.float64)

diff_ = rel_error(out, correct_out).item()

print()
assert diff_ < 1e-7, f'Task 6.4 failed: difference too large ({diff_:.6e})'

# Compare your output with ours. Difference should be around 1e-8.
print('Task 6.4 passed!')
print('Testing max_pool_forward_naive function:')
print('difference: ', diff_)

# Max pooling: Naive backward

**Task 6.5** 

Implement the backward pass for the max-pooling operation in the function `max_pool_backward_naive` in the file `conv_layers.py`. You don't need to worry about computational efficiency.

Check your implementation with numeric gradient checking by running the following:

In [ ]:
torch.manual_seed(231)

x = torch.randn(3, 2, 8, 8, dtype=torch.float64)
dout = torch.randn(3, 2, 4, 4, dtype=torch.float64)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

dx_num = eval_numerical_gradient_array(lambda x_: max_pool_forward_naive(x_, pool_param)[0], x, dout)

out, cache = max_pool_forward_naive(x, pool_param)
dx = max_pool_backward_naive(dout, cache)

dx_diff = rel_error(dx, dx_num).item()

print()
assert dx_diff < 1e-11, f'Task 6.5 failed: dx error too large ({dx_diff:.6e})'

# Your error should be around 1e-12
print('Task 6.5 passed!')
print('Testing max_pool_backward_naive function:')
print('dx error: ', dx_diff)

# Fast layers
Making convolution and pooling layers fast can be challenging. To spare you the pain, we've provided fast implementations of the forward and backward passes for convolution and pooling layers in the file `fast_layers.py`.

The API for the fast versions of the convolution and pooling layers is exactly the same as the naive versions that you implemented above: the forward pass receives data, weights, and parameters and produces outputs and a cache object; the backward pass recieves upstream derivatives and the cache object and produces gradients with respect to the data and weights.

**NOTE:** The fast implementation for pooling will only perform optimally if the pooling regions are non-overlapping and tile the input. If these conditions are not met then the fast pooling implementation will not be much faster than the naive implementation.

You can compare the performance of the naive and fast versions of these layers by running the following:

In [ ]:
from fast_layers import conv_forward_strides, conv_backward_strides
from time import time

torch.manual_seed(231)
x = torch.randn(100, 3, 31, 31)
w = torch.randn(25, 3, 3, 3)
b = torch.randn(25,)
dout = torch.randn(100, 25, 16, 16)
conv_param = {'stride': 2, 'pad': 1}

t0 = time()
out_naive, cache_naive = conv_forward_naive(x, w, b, conv_param)
t1 = time()
out_fast, cache_fast = conv_forward_strides(x, w, b, conv_param)
t2 = time()

print('Testing conv_forward_strides:')
print('Naive: %fs' % (t1 - t0))
print('Fast: %fs' % (t2 - t1))
print('Speedup: %fx' % ((t1 - t0) / (t2 - t1)))
print('Difference: ', rel_error(out_naive, out_fast))

t0 = time()
dx_naive, dw_naive, db_naive = conv_backward_naive(dout, cache_naive)
t1 = time()
dx_fast, dw_fast, db_fast = conv_backward_strides(dout, cache_fast)
t2 = time()

print('\nTesting conv_backward_strides:')
print('Naive: %fs' % (t1 - t0))
print('Fast: %fs' % (t2 - t1))
print('Speedup: %fx' % ((t1 - t0) / (t2 - t1)))
print('dx difference: ', rel_error(dx_naive, dx_fast))
print('dw difference: ', rel_error(dw_naive, dw_fast))
print('db difference: ', rel_error(db_naive, db_fast))

In [ ]:
from fast_layers import max_pool_forward_fast, max_pool_backward_fast

torch.manual_seed(231)

x = torch.randn(100, 3, 32, 32)
dout = torch.randn(100, 3, 16, 16)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

t0 = time()
out_naive, cache_naive = max_pool_forward_naive(x, pool_param)
t1 = time()
out_fast, cache_fast = max_pool_forward_fast(x, pool_param)
t2 = time()

print('Testing pool_forward_fast:')
print('Naive: %fs' % (t1 - t0))
print('fast: %fs' % (t2 - t1))
print('speedup: %fx' % ((t1 - t0) / (t2 - t1)))
print('difference: ', rel_error(out_naive, out_fast))

t0 = time()
dx_naive = max_pool_backward_naive(dout, cache_naive)
t1 = time()
dx_fast = max_pool_backward_fast(dout, cache_fast)
t2 = time()

print('\nTesting pool_backward_fast:')
print('Naive: %fs' % (t1 - t0))
print('speedup: %fx' % ((t1 - t0) / (t2 - t1)))
print('dx difference: ', rel_error(dx_naive, dx_fast))

# Convolutional "sandwich" layers
Previously we introduced the concept of "sandwich" layers that combine multiple operations into commonly used patterns. In the file `layer_utils.py` you will find sandwich layers that implement a few commonly used patterns for convolutional networks.

In [ ]:
from layer_utils import conv_relu_pool_forward, conv_relu_pool_backward

torch.manual_seed(231)

x = torch.randn(2, 3, 16, 16)
w = torch.randn(3, 3, 3, 3)
b = torch.randn(3,)
dout = torch.randn(2, 3, 8, 8)

conv_param = {'stride': 1, 'pad': 1}
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

out, cache = conv_relu_pool_forward(x, w, b, conv_param, pool_param)
dx, dw, db = conv_relu_pool_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], b, dout)

print('Testing conv_relu_pool')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

In [ ]:
from layer_utils import conv_relu_forward, conv_relu_backward

torch.manual_seed(231)

x = torch.randn(2, 3, 8, 8)
w = torch.randn(3, 3, 3, 3)
b = torch.randn(3,)
dout = torch.randn(2, 3, 8, 8)

conv_param = {'stride': 1, 'pad': 1}

out, cache = conv_relu_forward(x, w, b, conv_param)
dx, dw, db = conv_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: conv_relu_forward(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_relu_forward(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_relu_forward(x, w, b, conv_param)[0], b, dout)

print('Testing conv_relu:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# Three-layer ConvNet

Now that you have implemented all the necessary layers, we can put them together into a simple convolutional network.

**Task 6.6** 

Open the file `cnn.py` and complete the implementation of the `ThreeLayerConvNet` class. Run the following cells to help you debug:

## Sanity check loss
After you build a new network, one of the first things you should do is sanity check the loss. When we use the softmax loss, we expect the loss for random weights (and no regularization) to be about `log(C)` for `C` classes. When we add regularization this should go up.

In [ ]:
import math

model = ThreeLayerConvNet()

N = 50

X = torch.randn(N, 3, 32, 32)
y = torch.randint(10, size=(N,))

loss, grads = model.loss(X, y)

print()
assert math.isclose(loss.item(), torch.log(torch.tensor(10.0)).item(), rel_tol=1e-1), f'Task 6.6 failed: loss too high ({loss.item():.6f})'

print('Initial loss (no regularization): ', loss)

model.reg = 0.5
loss, grads = model.loss(X, y)

print()
assert math.isclose(loss.item(), torch.log(torch.tensor(10.0)).item(), rel_tol=1e-1), f'Task 6.6 failed: loss too high ({loss.item():.6f})'

print('Initial loss (with regularization): ', loss)

print('Task 6.6 passed!')

## Gradient check
After the loss looks reasonable, use numeric gradient checking to make sure that your backward pass is correct. When you use numeric gradient checking you should use a small amount of artifical data and a small number of neurons at each layer. Note: correct implementations may still have relative errors up to 1e-2.

In [ ]:
num_inputs = 2
input_dim = (3, 16, 16)
reg = 0.0
num_classes = 10

torch.manual_seed(231)
X = torch.randn(num_inputs, *input_dim)
y = torch.randint(num_classes, size=(num_inputs,))

model = ThreeLayerConvNet(num_filters=3, filter_size=3,
                          input_dim=input_dim, hidden_dim=7,
                          dtype=torch.float64)

loss, grads = model.loss(X, y)

for param_name in sorted(grads):

    f = lambda _: model.loss(X, y)[0]
    
    param_grad_num = eval_numerical_gradient(f, model.params[param_name], verbose=False, h=1e-6)
    e = rel_error(param_grad_num, grads[param_name])
    print('%s max relative error: %e' % (param_name, rel_error(param_grad_num, grads[param_name])))

## Overfit small data
A nice trick is to train your model with just a few training samples. You should be able to overfit small datasets, which will result in very high training accuracy and comparatively low validation accuracy.

In [ ]:
torch.manual_seed(231)

num_train = 100
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

model = ThreeLayerConvNet(weight_scale=1e-2)

solver = Solver(model, small_data,
                num_epochs=15, batch_size=50,
                update_rule='adam',
                optim_config={
                  'learning_rate': 1e-3,
                },
                verbose=True, print_every=1)
solver.train()

Plotting the loss, training accuracy, and validation accuracy should show clear overfitting:

In [ ]:
plt.subplot(2, 1, 1)
plt.plot(solver.loss_history, 'o')
plt.xlabel('iteration')
plt.ylabel('loss')

plt.subplot(2, 1, 2)
plt.plot(solver.train_acc_history, '-o')
plt.plot(solver.val_acc_history, '-o')
plt.legend(['train', 'val'], loc='upper left')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.show()

## Train the net

**Task 6.7**

By training the three-layer convolutional network for one epoch, you should achieve greater than 40% accuracy on the training set:

In [ ]:
model = ThreeLayerConvNet(weight_scale=0.001, hidden_dim=500, reg=0.001)

solver = Solver(model, data,
                num_epochs=1, batch_size=50,
                update_rule='adam',
                optim_config={
                  'learning_rate': 1e-3,
                },
                verbose=True, print_every=20)
solver.train()

print()
assert solver.train_acc_history[-1] > 0.4, f'Task 6.7 failed: final training accuracy too low ({solver.train_acc_history[-1]:.4f})'

print('Task 6.7 passed!')

## Visualize Filters
You can visualize the first-layer convolutional filters from the trained network by running the following:

In [ ]:
from vis_utils import visualize_grid

grid = visualize_grid(model.params['W1'].permute(0, 2, 3, 1).numpy())
plt.imshow(grid.astype('uint8'))
plt.axis('off')
plt.gcf().set_size_inches(5, 5)
plt.show()